# 🎒 Bag Detection — Swift-YOLO 320 → SenseCap A1102

This notebook trains a **Swift-YOLO** model on your COCO-format bag dataset,
exports and **INT8-quantizes** it to TFLite, then packages it as a `.uf2`
file ready to deploy on the **SenseCap A1102** via the SenseCraft app.

### Dataset location (Google Drive)
```
/content/drive/MyDrive/Detect bag/Bag.v1-bagtrain.coco/
  ├── train/
  ├── valid/
  ├── test/
  └── README files
```

> **Runtime:** Make sure you select **GPU** → Runtime → Change runtime type → T4 GPU

---
## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Verify your dataset exists ──────────────────────────────────────────────
DATASET_ROOT = '/content/drive/MyDrive/Detect bag/Bag.v1-bagtrain.coco'
TRAIN_ANN    = os.path.join(DATASET_ROOT, 'train/_annotations.coco.json')
VALID_ANN    = os.path.join(DATASET_ROOT, 'valid/_annotations.coco.json')

for path in [DATASET_ROOT, TRAIN_ANN, VALID_ANN]:
    status = '✅ FOUND' if os.path.exists(path) else '❌ MISSING'
    print(f'{status}: {path}')

---
## Cell 2 — Install SSCMA (Seeed SenseCraft Model Assistant)

In [ ]:
# Install SSCMA and its dependencies
!pip install -q sscma

# Verify installation
import sscma
print(f'✅ SSCMA version: {sscma.__version__}')

---
## Cell 3 — Inspect Dataset & Extract Class Names

In [ ]:
import json

with open(TRAIN_ANN) as f:
    coco_data = json.load(f)

categories = coco_data['categories']
NUM_CLASSES = len(categories)
CLASS_NAMES = [c['name'] for c in categories]

print(f'📦 Number of classes : {NUM_CLASSES}')
print(f'📝 Class names       : {CLASS_NAMES}')
print(f'🖼️  Training images   : {len(coco_data["images"])}')
print(f'🏷️  Training annotations: {len(coco_data["annotations"])}')

---
## Cell 4 — Configuration

Adjust `EPOCHS` and `BATCH_SIZE` to your needs.
* **320×320** input is the correct size for SenseCap A1102 (Grove Vision AI V2 / Himax WiseEye2 backend).
* More epochs → better accuracy but longer training.

In [ ]:
# ── User-editable settings ──────────────────────────────────────────────────
EPOCHS      = 100     # increase for better accuracy (e.g. 200-300)
BATCH_SIZE  = 16
IMG_SIZE    = 320     # A1102 target input resolution
WORK_DIR    = 'Bag_Detection_Swift-YOLO_320'

# ── Derived paths (do not edit) ─────────────────────────────────────────────
PRETRAIN_URL = (
    'https://github.com/Seeed-Studio/SSCMA/releases/download/'
    'model_zoo/swift_yolo_tiny_coco_pretrain.pth'
)
PRETRAIN_PATH = f'{WORK_DIR}/pretrain.pth'

os.makedirs(WORK_DIR, exist_ok=True)
print(f'Work directory: {WORK_DIR}')
print(f'Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Image size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

---
## Cell 5 — Download Pretrained Weights

In [ ]:
import urllib.request

if not os.path.exists(PRETRAIN_PATH):
    print('⬇️  Downloading pretrained weights...')
    urllib.request.urlretrieve(PRETRAIN_URL, PRETRAIN_PATH)
    print(f'✅ Saved to {PRETRAIN_PATH}')
else:
    print(f'✅ Pretrained weights already exist at {PRETRAIN_PATH}')

---
## Cell 6 — Symlink Dataset into Work Directory

SSCMA expects `data_root` to contain `train/`, `valid/`, and their `_annotations.coco.json` files.

In [ ]:
DATASET_LINK = f'{WORK_DIR}/dataset'

if not os.path.exists(DATASET_LINK):
    os.symlink(DATASET_ROOT, DATASET_LINK)
    print(f'🔗 Linked dataset → {DATASET_LINK}')
else:
    print(f'✅ Dataset link already exists')

# Quick sanity check
!ls -la {DATASET_LINK}

---
## Cell 7 — 🚀 Train the Model

This launches Swift-YOLO training with your bag dataset.
* `swift_yolo_tiny_1xb16_300e_coco.py` is the base config (overridden below)
* Training logs are saved to `{WORK_DIR}/`
* Monitor with the printed loss/mAP lines every epoch

> ⏱️ ~30 min for 100 epochs on T4 GPU

In [ ]:
!sscma.train configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
    --cfg-options \
    work_dir={WORK_DIR} \
    num_classes={NUM_CLASSES} \
    epochs={EPOCHS} \
    height={IMG_SIZE} \
    width={IMG_SIZE} \
    batch={BATCH_SIZE} \
    data_root={WORK_DIR}/dataset/ \
    load_from={PRETRAIN_PATH}

---
## Cell 8 — Find Best Checkpoint

In [ ]:
import glob

# Check last_checkpoint file (SSCMA writes the best path here)
last_ckpt_file = f'{WORK_DIR}/last_checkpoint'

if os.path.exists(last_ckpt_file):
    with open(last_ckpt_file) as f:
        BEST_CHECKPOINT = f.read().strip()
    print(f'✅ Best checkpoint: {BEST_CHECKPOINT}')
else:
    # Fallback: find most recent .pth
    pth_files = sorted(glob.glob(f'{WORK_DIR}/*.pth'))
    BEST_CHECKPOINT = pth_files[-1] if pth_files else None
    print(f'✅ Using checkpoint: {BEST_CHECKPOINT}')

assert BEST_CHECKPOINT and os.path.exists(BEST_CHECKPOINT), \
    '❌ No checkpoint found — did training complete successfully?'

---
## Cell 9 — Export & INT8 Quantize to TFLite

The A1102 (Himax WiseEye2) requires a **TFLite INT8** model.
SSCMA handles quantization automatically using the training data as calibration set.

In [ ]:
EXPORT_DIR = f'{WORK_DIR}/export'
os.makedirs(EXPORT_DIR, exist_ok=True)

!sscma.export configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
    {BEST_CHECKPOINT} \
    --cfg-options \
    work_dir={EXPORT_DIR} \
    num_classes={NUM_CLASSES} \
    height={IMG_SIZE} \
    width={IMG_SIZE} \
    data_root={WORK_DIR}/dataset/ \
    --target tflite \
    --precision int8 \
    --quant_type coco

---
## Cell 10 — Locate the Quantized TFLite File

In [ ]:
tflite_files = glob.glob(f'{EXPORT_DIR}/**/*.tflite', recursive=True)

print('Found TFLite files:')
for f in tflite_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'  📁 {f}  ({size_kb:.1f} KB)')

# Prefer int8 variant
int8_files = [f for f in tflite_files if 'int8' in f.lower()]
TFLITE_MODEL = int8_files[0] if int8_files else (tflite_files[0] if tflite_files else None)

assert TFLITE_MODEL, '❌ No TFLite file found — did export succeed?'
print(f'\n✅ Using: {TFLITE_MODEL}')

---
## Cell 11 — Package as UF2 for A1102

The SenseCap A1102 uses the **Grove Vision AI V2 / Himax WiseEye2** chip.
We convert the TFLite model to a `.uf2` file that can be flashed via SenseCraft.

In [ ]:
# Download the uf2conv tool for Grove Vision AI V2 (Himax WE2)
!wget -q https://raw.githubusercontent.com/Seeed-Studio/SSCMA-Micro/main/scripts/uf2conv.py -O uf2conv.py
!wget -q https://raw.githubusercontent.com/Seeed-Studio/SSCMA-Micro/main/scripts/uf2families.json -O uf2families.json

UF2_OUTPUT = f'{WORK_DIR}/bag_detection_model.uf2'

# -f GROVEAI  → targets the Grove Vision AI V2 / Himax WE2 family
# -t 1        → model slot 1 (you can use 1-4)
!python uf2conv.py \
    -f GROVEAI \
    -t 1 \
    -c {TFLITE_MODEL} \
    -o {UF2_OUTPUT}

if os.path.exists(UF2_OUTPUT):
    size_kb = os.path.getsize(UF2_OUTPUT) / 1024
    print(f'\n✅ UF2 created: {UF2_OUTPUT}  ({size_kb:.1f} KB)')
else:
    print('❌ UF2 creation failed — check logs above')

---
## Cell 12 — Save All Outputs to Google Drive

In [ ]:
import shutil

GDRIVE_OUT = '/content/drive/MyDrive/Detect bag/trained_model'
os.makedirs(GDRIVE_OUT, exist_ok=True)

# Copy UF2
shutil.copy(UF2_OUTPUT, f'{GDRIVE_OUT}/bag_detection_model.uf2')
print(f'✅ UF2   → {GDRIVE_OUT}/bag_detection_model.uf2')

# Copy TFLite
shutil.copy(TFLITE_MODEL, f'{GDRIVE_OUT}/bag_detection_int8.tflite')
print(f'✅ TFLite → {GDRIVE_OUT}/bag_detection_int8.tflite')

# Copy best checkpoint
shutil.copy(BEST_CHECKPOINT, f'{GDRIVE_OUT}/best_checkpoint.pth')
print(f'✅ Checkpoint → {GDRIVE_OUT}/best_checkpoint.pth')

print('\n🎉 All files saved to Google Drive!')

---
## Cell 13 — (Optional) Download UF2 Directly from Colab

In [ ]:
from google.colab import files
files.download(UF2_OUTPUT)

---
## Cell 14 — (Optional) Evaluate the Exported Model

Run inference on the validation set to check mAP of the quantized TFLite model.

In [ ]:
!sscma.inference configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
    {TFLITE_MODEL} \
    --cfg-options \
    num_classes={NUM_CLASSES} \
    height={IMG_SIZE} \
    width={IMG_SIZE} \
    data_root={WORK_DIR}/dataset/ \
    --dump_dir {WORK_DIR}/inference_results

---
## 📱 Deploying to SenseCap A1102 via SenseCraft App

### Option A — Flash via USB (Direct)
1. Connect your **SenseCap A1102** to your PC with a USB-C cable  
2. **Double-click** the BOOT button on the device to enter mass-storage mode  
3. A drive called `SENSECAP` will appear on your file explorer  
4. Drag & drop `bag_detection_model.uf2` onto the `SENSECAP` drive  
5. The drive will disappear — the model is now flashed ✅

### Option B — Deploy via SenseCraft AI Platform (No-code)
1. Go to **https://sensecraft.seeed.cc**  
2. Sign in → **My Models** → **Upload Model**  
3. Upload `bag_detection_int8.tflite`  
4. Set class names (e.g., `bag`) and model type to `Swift-YOLO 320`  
5. Select your A1102 device and click **Deploy**  
6. Open the **SenseCraft app** on your phone to monitor live detections

### Recommended SenseCraft App Settings
| Setting | Value |
|---|---|
| Confidence threshold | 0.45–0.55 |
| IoU threshold | 0.45 |
| Model slot | 1 |

---
> **Tips for better accuracy:**
> - Increase `EPOCHS` to 200–300 if mAP is low
> - Capture images at 1–5 m distance from bags
> - Make sure the A1102 camera faces the bags directly with good lighting
> - Add more diverse training images (different angles, lighting, backgrounds)